In [1]:
import sys
import numpy as np 
from pathlib import Path
import os
import torch

In [2]:
# add parent directory to path
sys.path.append(str(Path('..').resolve()))

In [3]:
import nnetflow as nnf

In [4]:
nnf?

Type:        module
String form: <module 'nnetflow' from '/home/njue/.projects/nnetflow/nnetflow/__init__.py'>
File:        ~/.projects/nnetflow/nnetflow/__init__.py
Docstring:   <no docstring>

in this notebook i want to test the performance of diffrent operations on tensors and see if they are first enough or we can make some imporvement, and we will also be comparing that to pytorch :)

In [5]:
t = np.random.rand(1000, 1000)
t_nnetflow = nnf.Tensor(t, requires_grad=True)
t_torch = torch.tensor(t, requires_grad=True)

In [6]:
t_nnetflow.shape, t_torch.shape

((1000, 1000), torch.Size([1000, 1000]))

In [7]:
# let profile the performance of nnetflow and torch for a simple operation for diffrent operations , 
## ======================== ADDITION =========================

a  = nnf.Tensor(np.arange(1000000), requires_grad=False)
b = nnf.Tensor(np.arange(1000000), requires_grad=False)
%timeit c = a + b

2.74 ms ± 125 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [8]:
a_t = torch.tensor(np.arange(1000000), requires_grad=False)
b_t = torch.tensor(np.arange(1000000), requires_grad=False)
%timeit c_t = a_t + b_t

3.26 ms ± 289 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [10]:
# am going to  create a small network and profile the performance of nnetflow and torch 

## nnetflow network 

# inputs -> 1000 features
# outpus  -> 10 classes 

input_size  = 1000
output_size = 10
# generate random input data , i will have 10000 samples

X = np.random.rand(10000, input_size)
y = np.random.randint(0, output_size, size=(10000,))

In [13]:
nnf_hidden_layer1 = nnf.Tensor(np.random.rand(input_size, 128), requires_grad=True)
nnf_hidden_layer2 = nnf.Tensor(np.random.rand(128, 64), requires_grad=True)
nnf_output_layer = nnf.Tensor(np.random.rand(64, output_size), requires_grad=True)

torch_hidden_layer1 = torch.tensor(np.random.rand(input_size, 128), requires_grad=True)
torch_hidden_layer2 = torch.tensor(np.random.rand(128, 64), requires_grad=True)
torch_output_layer = torch.tensor(np.random.rand(64, output_size), requires_grad=True)

In [20]:
%%timeit
X_nnf = nnf.Tensor(X, requires_grad=False)
intermediate_1 = X_nnf.matmul(nnf_hidden_layer1) # [batch_size, input_size] x [input_size, 128] = [batch_size, 128]
intermediate_2 = intermediate_1.relu() # shape dont change
intermediate_3 = intermediate_2.matmul(nnf_hidden_layer2) # [batch_size, 128] x [128, 64] = [batch_size, 64]
intermediate_4 = intermediate_3.relu() # shape dont change
output_nnf = intermediate_4.matmul(nnf_output_layer) # [batch_size, 64] x [64, output_size] = [batch_size, output_size]


187 ms ± 101 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
## nnetflow forward pass
%%timeit    
X_torch = torch.tensor(X, requires_grad=False)
intermediate_1_torch = X_torch.matmul(torch_hidden_layer1) # [batch_size, input_size] x [input_size, 128] = [batch_size, 128]
intermediate_2_torch = intermediate_1_torch.relu() # shape dont change
intermediate_3_torch = intermediate_2_torch.matmul(torch_hidden_layer2) # [batch_size, 128] x [128, 64] = [batch_size, 64]
intermediate_4_torch = intermediate_3_torch.relu() # shape dont change
output_torch = intermediate_4_torch.matmul(torch_output_layer) # [batch_size, 64] x [64, output_size] = [batch_size, output_size]

112 ms ± 7.49 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [22]:
%pip install memory_profiler

/home/njue/.projects/nnetflow/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
## check the memory usage: 
%load_ext memory_profiler

In [25]:
%%memit
X_torch = torch.tensor(X, requires_grad=False)
intermediate_1_torch = X_torch.matmul(torch_hidden_layer1) # [batch_size, input_size] x [input_size, 128] = [batch_size, 128]
intermediate_2_torch = intermediate_1_torch.relu() # shape dont change
intermediate_3_torch = intermediate_2_torch.matmul(torch_hidden_layer2) # [batch_size, 128] x [128, 64] = [batch_size, 64]
intermediate_4_torch = intermediate_3_torch.relu() # shape dont change
output_torch = intermediate_4_torch.matmul(torch_output_layer) # [batch_size, 64] x [64, output_size] = [batch_size, output_size]

peak memory: 1151.61 MiB, increment: 0.00 MiB


In [26]:
%%memit
X_nnf = nnf.Tensor(X, requires_grad=False)
intermediate_1 = X_nnf.matmul(nnf_hidden_layer1) # [batch_size, input_size] x [input_size, 128] = [batch_size, 128]
intermediate_2 = intermediate_1.relu() # shape dont change
intermediate_3 = intermediate_2.matmul(nnf_hidden_layer2) # [batch_size, 128] x [128, 64] = [batch_size, 64]
intermediate_4 = intermediate_3.relu() # shape dont change
output_nnf = intermediate_4.matmul(nnf_output_layer) # [batch_size, 64] x [64, output_size] = [batch_size, output_size]


peak memory: 1080.55 MiB, increment: 5.17 MiB
